# Local CPU-Safe Checks

Use this notebook on the local HP laptop. It assumes the repository and `data/` folder are already present locally.

This notebook is CPU-safe: it validates data, rebuilds metadata/splits, and runs a tiny dataloader dry-run. It does not train ResNet-50, extract full embeddings, or run any long image pass.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# If this notebook is opened from the repo, this resolves to the repo root.
# If needed, set PROJECT_ROOT manually to your local project path.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Local CPU-safe configuration", flush=True)
print("- Python:", sys.version, flush=True)
print("- project root:", PROJECT_ROOT, flush=True)
print("- data dir:", PROJECT_ROOT / "data", flush=True)
print("- outputs dir:", PROJECT_ROOT / "outputs", flush=True)

os.chdir(PROJECT_ROOT)


## 1. Runtime Check


In [ ]:
try:
    import torch
    print("Torch:", torch.__version__, flush=True)
    print("CUDA:", torch.cuda.is_available(), flush=True)
    print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", flush=True)
except Exception as exc:
    print("Torch check failed:", repr(exc), flush=True)
    print("Install dependencies from PowerShell with: python -m pip install -r requirements.txt", flush=True)


## 2. Check Required Files


In [ ]:
print("Step 2: checking local files", flush=True)
required = [
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "scripts/01_make_metadata_csv.py",
    PROJECT_ROOT / "scripts/02_make_image_splits.py",
    PROJECT_ROOT / "scripts/03_train_resnet50_image_only.py",
    PROJECT_ROOT / "data/processed/concepts.csv",
    PROJECT_ROOT / "data/processed/images.csv",
    PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGS/object_images",
    PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0",
]
missing = []
for path in required:
    exists = path.exists()
    print(("ok:      " if exists else "missing: ") + str(path), flush=True)
    if not exists:
        missing.append(path)
if missing:
    raise FileNotFoundError("Missing required local files/folders. See printed paths above.")
print("Required local files are present.", flush=True)


## 3. Verify Processed Data


In [ ]:
import pandas as pd

print("Step 3: verifying processed CSVs and image folders", flush=True)
concepts = pd.read_csv(PROJECT_ROOT / "data/processed/concepts.csv")
images = pd.read_csv(PROJECT_ROOT / "data/processed/images.csv")

things_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGS/object_images"
cc0_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0"
things_count = sum(1 for _ in things_dir.rglob("*.jpg"))
cc0_count = sum(1 for _ in cc0_dir.rglob("*.jpg"))

print("concepts rows:", len(concepts), flush=True)
print("concept_index unique:", concepts["concept_index"].nunique(), flush=True)
print("images rows:", len(images), flush=True)
print("image concept_index nulls:", int(images["concept_index"].isna().sum()), flush=True)
print("image concept_index unique:", images["concept_index"].nunique(), flush=True)
print("THINGS jpg count:", things_count, flush=True)
print("THINGSplus CC0 jpg count:", cc0_count, flush=True)

assert len(concepts) == 1854
assert concepts["concept_index"].nunique() == 1854
assert images["concept_index"].isna().sum() == 0
assert images["concept_index"].nunique() == 1854
assert things_count == 26107
print("Processed data checks passed.", flush=True)


## 4. Rebuild Metadata and Splits


In [ ]:
print("Step 4: rebuilding image metadata and splits", flush=True)
subprocess.run([sys.executable, "scripts/01_make_metadata_csv.py"], check=True)
subprocess.run([sys.executable, "scripts/02_make_image_splits.py"], check=True)

metadata = pd.read_csv(PROJECT_ROOT / "data/baseline/image_metadata.csv")
splits = pd.read_csv(PROJECT_ROOT / "data/baseline/image_splits.csv")
print("metadata rows:", len(metadata), flush=True)
print("metadata concepts:", metadata["concept_id"].nunique(), flush=True)
print("missing metadata images:", int((~metadata["image_exists"].astype(bool)).sum()), flush=True)
print("split rows:", len(splits), flush=True)
print("split counts:", splits["split"].value_counts().to_dict(), flush=True)
print("concepts per split:", splits.groupby("split")["concept_id"].nunique().to_dict(), flush=True)

assert metadata["concept_id"].nunique() == 1854
assert splits["concept_id"].nunique() == 1854
assert int((~splits["image_exists"].astype(bool)).sum()) == 0
print("Metadata and split checks passed.", flush=True)


## 5. Tiny Dataloader Dry-Run

This loads a few images through the dataset/transform path. It does not train.


In [ ]:
print("Step 5: running tiny dataloader dry-run", flush=True)
subprocess.run([
    sys.executable,
    "scripts/03_train_resnet50_image_only.py",
    "--dry-run",
    "--batch-size", "8",
    "--num-workers", "0",
], check=True)
print("Tiny dataloader dry-run passed.", flush=True)


## 6. Inspect Local Outputs


In [ ]:
print("Step 6: inspecting local outputs", flush=True)
paths = [
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "outputs/image_metadata_report.json",
    PROJECT_ROOT / "outputs/image_splits_report.json",
    PROJECT_ROOT / "outputs/baseline_resnet50/best_model.pt",
    PROJECT_ROOT / "outputs/baseline_resnet50/metrics.json",
    PROJECT_ROOT / "outputs/baseline_resnet50/training_log.csv",
]
for path in paths:
    if path.exists():
        size = path.stat().st_size if path.is_file() else 0
        print(f"exists: {path} size={size}", flush=True)
    else:
        print(f"missing: {path}", flush=True)

for report_path in [PROJECT_ROOT / "outputs/image_metadata_report.json", PROJECT_ROOT / "outputs/image_splits_report.json"]:
    if report_path.exists():
        print("\n", report_path.name, flush=True)
        print(report_path.read_text()[:2000], flush=True)


## 7. Stop Here on CPU

The CPU baseline training has completed. For future reruns, prefer a CUDA machine if available; otherwise run the core scripts locally from PowerShell.
